# Project19 Database Viewer

This notebook is for looking at the database visually: annotated images and Alpamayo prediction graphs.

## Setup

In [ ]:
import importlib
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
pipeline_dir = cwd if cwd.name == "pipeline" else cwd / "pipeline"
sys.path.insert(0, str(pipeline_dir))

import notebook_helpers
importlib.reload(notebook_helpers)
from notebook_helpers import open_explorer

db = open_explorer()

# Set this to one source/segment, like "route_1_segment_00".
# Leave it as None to use the whole database.
SOURCE_ID = None

## Annotated Images

In [ ]:
db.show_gallery(source=SOURCE_ID, limit=6)

## Pedestrian Annotations

In [ ]:
PEDESTRIAN_LIMIT = 6

pedestrian_rows = db.conn.execute(
    """
    SELECT DISTINCT f.id
    FROM frames f
    JOIN annotations a ON a.frame_id = f.id
    JOIN label_categories lc ON lc.annotation_id = a.id
    WHERE (? IS NULL OR f.source = ?)
      AND lc.category = 'pedestrian'
      AND lc.present = 1
    ORDER BY f.source, f.frame_number
    LIMIT ?
    """,
    (SOURCE_ID, SOURCE_ID, PEDESTRIAN_LIMIT),
).fetchall()

if not pedestrian_rows:
    print("No pedestrian annotations found.")

for row in pedestrian_rows:
    db.show_frame(int(row["id"]), figsize=(10, 5))

## Left/Right Turn Predictions

In [ ]:
TURN_LIMIT = 3

for turn_word in ["left", "right"]:
    turn_rows = db.conn.execute(
        """
        SELECT ap.id
        FROM alpamayo_predictions ap
        JOIN frames f ON f.id = ap.frame_id
        WHERE (? IS NULL OR f.source = ?)
          AND LOWER(ap.nav_command) LIKE ?
        ORDER BY f.source, f.frame_number, ap.id
        LIMIT ?
        """,
        (SOURCE_ID, SOURCE_ID, f"%{turn_word}%", TURN_LIMIT),
    ).fetchall()

    if not turn_rows:
        print(f"No {turn_word} predictions found.")

    for row in turn_rows:
        db.plot_prediction(int(row["id"]))